# Aula 03 — Regressão linear por OLS (Versão professor)

Notebook de condução docente para a terceira aula do treinamento inferencial.

## Finalidade desta versão

Esta edição foi pensada para o professor e, por isso, contém:

- explicações conceituais sobre ajuste por mínimos quadrados ordinários;
- notas de condução oral sobre interpretação de coeficientes;
- perguntas para leitura crítica do modelo;
- alertas contra uso mecânico do R²;
- conexão entre amostra saneada, ajuste e leitura dos resíduos.

## Resultado esperado da aula

Ao final da condução, a turma deve compreender que regressão não é uma caixa-preta: ela produz uma equação interpretável, com coeficientes, significância, graus de liberdade e erros que precisam ser lidos tecnicamente.

## Roteiro sugerido de tempo

- **0 a 6 min** — retomada da amostra saneada e motivo para modelagem;
- **6 a 15 min** — escolha da variável dependente e das explicativas;
- **15 a 28 min** — ajuste do modelo OLS;
- **28 a 38 min** — leitura da equação e dos coeficientes;
- **38 a 46 min** — leitura de R², R² ajustado, RMSE e teste F;
- **46 a 50 min** — introdução aos resíduos e ponte para a Aula 4.

## Estratégia didática

Nesta aula, o professor deve sustentar quatro mensagens centrais:

1. o modelo responde a uma pergunta econômica, não apenas estatística;
2. coeficiente tem sinal, magnitude e interpretação contextual;
3. R² sozinho não valida modelo;
4. resíduos são parte da leitura, não um detalhe descartável.

In [ ]:
# Importações da aula.
# Professor: este é um bom ponto para explicar a continuidade do curso.
# A base é carregada, a coluna de valor unitário pode ser recomposta,
# e o núcleo desta aula passa a ser o serviço de regressão.

from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import load_raw_dataset, resolve_project_root
from servicos.unitarizacao import add_unit_price_column
from servicos.regressao import (
    attach_predictions_and_residuals,
    build_coefficients_table,
    build_model_summary,
    fit_ols_regression,
)

## Nota de condução oral

Sugestão de fala:

“Se a Aula 1 colocou os dados em uma escala comparável e a Aula 2 melhorou a consistência da massa amostral, a Aula 3 tenta explicar o preço a partir das características do imóvel. Aqui começa propriamente a modelagem inferencial.”

In [ ]:
# Mesma estratégia de localização da base das aulas anteriores.
# Isso preserva coerência didática e reduz ruído de infraestrutura.

DATASET_CANDIDATES: Final[tuple[str, ...]] = (
    'amostras_residencial35.csv',
    'amostrasresidencial35.csv',
    'amostras_residencial.csv',
)

TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_default_dataset(project_root: Path) -> Path:
    """Localiza automaticamente a base padrão das aulas iniciais."""
    data_dir = project_root / 'data'

    for filename in DATASET_CANDIDATES:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate

    searched = ', '.join(DATASET_CANDIDATES)
    raise FileNotFoundError(
        'Nenhum arquivo padrão foi encontrado na pasta `data`. '
        f'Arquivos procurados: {searched}.'
    )

## Etapa 1 — Resolver a base da aula

### Intenção docente

Antes de modelar, confirme a infraestrutura e a base de entrada.
Isso ajuda a turma a entender que modelagem confiável depende tanto de especificação estatística quanto de organização correta dos dados.

In [ ]:
project_root = resolve_project_root()
dataset_path = locate_default_dataset(project_root)

project_root, dataset_path

## Etapa 2 — Carregar a base e preparar contexto

### O que dizer

Nesta aula, o preço total será tratado como variável dependente.
As variáveis explicativas escolhidas representam atributos que, em tese, ajudam a explicar economicamente o preço observado.

In [ ]:
df_raw = load_raw_dataset(dataset_path)
df_model = add_unit_price_column(df_raw)

print('Dimensão da base:', df_model.shape)
df_model.head()

### Nota ao professor

Se a turma perguntar por que modelar `preco` e não `valor_unitario`, use a pergunta como gancho didático.
Ela é válida e ajuda a discutir diferentes especificações possíveis de modelo.
Nesta implementação, seguimos a lógica do script da Aula 3, que usa `preco` como alvo principal. [cite:66]

## Etapa 3 — Escolher as variáveis explicativas disponíveis

### Intenção pedagógica

Nem toda variável desejada estará sempre presente na base.
Por isso, a seleção abaixo filtra automaticamente as variáveis preferidas que realmente existem no dataset.

In [ ]:
available_features = [column for column in PREFERRED_FEATURES if column in df_model.columns]

print('Variável dependente:', TARGET_COLUMN)
print('Variáveis explicativas disponíveis:', available_features)

### Pergunta para a turma

“Uma variável economicamente desejável, mas ausente na base, pode ser inventada na modelagem?”

## Etapa 4 — Ajustar o modelo OLS

### Mensagem-chave

O ajuste por mínimos quadrados ordinários procura os coeficientes que minimizam a soma dos quadrados dos resíduos.
Em linguagem de sala: estamos encontrando a equação linear que melhor aproxima os preços observados segundo as variáveis disponíveis.

In [ ]:
model = fit_ols_regression(
    df=df_model,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
)

model

### Nota de condução oral

Evite dizer apenas “o Python rodou a regressão”.
Diga algo como:

“Agora nós estimamos uma equação de preço. O software faz a conta, mas a leitura econômica e técnica continua sendo nossa responsabilidade.”

## Etapa 5 — Construir o resumo do modelo

### Objetivo docente

Aqui a turma passa do objeto matemático para a leitura interpretativa.
O resumo deve ajudar a responder: quantos dados entraram, quantos parâmetros foram estimados, quanto o modelo explica e qual é o erro típico de ajuste.

In [ ]:
summary = build_model_summary(model)
summary

## Etapa 6 — Tabela de coeficientes

### O que destacar

Cada coeficiente deve ser lido em três dimensões:

- **sinal**: positivo ou negativo;
- **magnitude**: quanto altera o preço, ceteris paribus;
- **significância**: quão consistente é sua contribuição estatística individual.

In [ ]:
coeff_table = build_coefficients_table(model)
coeff_table

### Fala sugerida

“Um coeficiente positivo indica associação positiva, e um coeficiente negativo indica associação negativa, mantidas as demais variáveis constantes. Mas o sinal, sozinho, não basta: precisamos ver magnitude e p-valor.”

## Etapa 7 — Montar uma forma legível da equação estimada

### Intenção didática

Transformar a tabela de coeficientes em equação textual ajuda a turma a perceber que regressão gera uma função explícita de preço.

In [ ]:
def resolve_equation_terms(coeff_table: pd.DataFrame) -> str:
    """Gera uma representação textual da equação estimada."""
    pieces: list[str] = []

    for _, row in coeff_table.iterrows():
        variable = str(row['variavel'])
        coefficient = float(row['coeficiente'])

        if variable == 'const':
            pieces.append(f'{coefficient:,.2f}')
            continue

        signal = '+' if coefficient >= 0 else '-'
        pieces.append(f'{signal} {abs(coefficient):,.2f}·{variable}')

    return 'Preço = ' + ' '.join(pieces)


equation = resolve_equation_terms(coeff_table)
print(equation)

### Pergunta para a turma

“Se a área privativa aumenta uma unidade, o que acontece com o preço estimado, mantidas as demais variáveis constantes?”

## Etapa 8 — Ler métricas globais do modelo

### Ponto conceitual

O professor deve separar claramente as métricas:

- **R²**: proporção explicada pelo modelo;
- **R² ajustado**: versão penalizada pela quantidade de variáveis;
- **RMSE**: tamanho típico do erro;
- **Teste F**: significância global do modelo.

O erro didático mais comum aqui é transformar R² em sinônimo de qualidade total do modelo.

In [ ]:
print(f"Observações usadas no ajuste: {summary['n_obs']}")
print(f"Parâmetros estimados: {summary['n_params']}")
print(f"Graus de liberdade do modelo: {summary['gl_modelo']}")
print(f"Graus de liberdade dos resíduos: {summary['gl_residuos']}")
print(f"R²: {summary['r2']:.4f}")
print(f"R² ajustado: {summary['r2_ajustado']:.4f}")
print(f"RMSE: {summary['rmse']:,.2f}")
print(f"Teste F: {summary['f_statistic']:.4f}")
print(f"p-valor do teste F: {summary['f_p_value']:.6f}")

### Nota ao professor

Use este momento para dizer explicitamente:

- R² alto não salva especificação ruim;
- R² baixo não inutiliza automaticamente um modelo aplicado;
- RMSE traz a escala do erro para uma unidade economicamente mais intuitiva;
- teste F ajuda a saber se o modelo, como conjunto, tem relevância estatística.

## Etapa 9 — Acrescentar valores ajustados e resíduos

### Finalidade pedagógica

Esta etapa prepara a Aula 4.
Ao anexar predições e resíduos, a turma começa a enxergar onde o modelo acerta melhor e onde erra mais.

In [ ]:
enriched_df = attach_predictions_and_residuals(df_model, model)
enriched_df.head()

In [ ]:
preview_columns = [
    column
    for column in ('id', 'preco', 'valor_ajustado', 'residuo')
    if column in enriched_df.columns
]

enriched_df[preview_columns].head(10)

### Fala sugerida

“O resíduo é a parte do preço que o modelo não explicou. Na próxima aula, nós vamos avaliar se esse comportamento residual respeita as condições esperadas para uma boa inferência.”

## Etapa 10 — Leitura crítica inicial do ajuste

### Perguntas orientadoras

- os sinais dos coeficientes fazem sentido econômico?
- há variáveis com baixa significância individual?
- o modelo parece globalmente significativo?
- o erro típico parece aceitável para o problema estudado?
- os resíduos sugerem necessidade de diagnóstico mais profundo?

## Erros conceituais comuns

- acreditar que regressão substitui raciocínio técnico;
- interpretar coeficiente sem considerar unidade e contexto;
- usar apenas R² para avaliar o modelo;
- esquecer graus de liberdade;
- ignorar resíduos porque a equação “parece boa”.

## Exercício supervisionado

Peça à turma que redija, com base nas saídas do notebook:

- uma frase explicando o papel da variável dependente;
- uma frase interpretando um coeficiente escolhido;
- uma frase comentando o significado do teste F;
- uma frase justificando por que a análise ainda não terminou na Aula 3.

In [ ]:
# Espaço livre para exploração guiada.
# Professor: use esta célula para pedir que a turma filtre casos,
# compare resíduos ou discuta sinais inesperados.

enriched_df.sort_values(by='residuo', ascending=False).head(10)

## Ponte para a Aula 4

Encerre com esta transição:

“Hoje nós ajustamos a equação e começamos a ler seus resultados. Na próxima aula, vamos verificar se os resíduos e os testes estatísticos sustentam tecnicamente o uso desse modelo.”